# 03 - Agente RAG: base normativa, recuperação, avaliação e logs

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
from IPython.display import Image, display, Markdown
pd.set_option('display.max_columns', 40); pd.set_option('display.width', 200)
fig = lambda n: display(Image(filename=str(ROOT / 'reports/figures' / n)))
met = lambda n: pd.read_csv(ROOT / 'reports/metrics' / n)

In [ ]:
import collections
from src.rag.retrieve import load_chunks
ch = load_chunks(); print(len(ch), 'chunks')
pd.Series(collections.Counter(c['sigla'] for c in ch)).rename('chunks').to_frame()

Exemplo de chunk (unidade = artigo, com dispositivos citáveis):

In [ ]:
c = next(c for c in ch if c['doc_id']=='conama_491_2018' and c['label'].startswith('art.')); print(json.dumps({k: c[k] for k in ['id','sigla','path','label','dispositivos','vigencia_inicio','vigencia_fim']}, ensure_ascii=False, indent=1)); print(c['text'][:1200])

## Benchmark de recuperação (recall@k, MRR)

In [ ]:
met('rag_retrieval.csv').round(3)

## Avaliação ponta a ponta (acerto, citação, fidelidade, recusa, ferramenta)

In [ ]:
met('rag_eval_resumo.csv').round(3).T

In [ ]:
import glob
for f in sorted(glob.glob('reports/metrics/rag_eval_detalhe_*.csv'))[:1]:
    d = pd.read_csv(f); display(Markdown(f'`{f}`')); display(d.drop(columns=['resposta']).round(2))

## Exemplo de execução registrada (pergunta, trechos, ferramentas, resposta, tempo, custo)

In [ ]:
runs = [json.loads(l) for l in open('logs/agent_runs.jsonl', encoding='utf-8')]
r = next((r for r in runs if any(t['ferramenta']=='estimar_pm25' for t in r['ferramentas'])), runs[-1])
print(r['pergunta']); print(); print(r['resposta']); print()
print(json.dumps({k: r[k] for k in ['tempo_total_s','tokens_entrada','tokens_saida','custo_estimado_usd','trechos','validacao']}, ensure_ascii=False, indent=1))